In [2]:
import json, os, pathlib
if pathlib.Path.cwd().name == "notebooks":
    os.chdir("..")          # repo 루트로 이동
sets = [json.loads(l) for l in open("data/metadata/beatmapsets.jsonl", encoding="utf-8")]
print("총 세트:", len(sets))

# 4K 난이도가 하나라도 있는 세트 수 — 필드명은 sample_response.json 메모 기준!
def has_4k(s):
    return any(b.get("mode_int") == 3 and b.get("cs") == 4 for b in s.get("beatmaps", []))
four_k = [s for s in sets if has_4k(s)]
print("4K 포함 세트:", len(four_k))

# 중복 확인
ids = [s["id"] for s in sets]
print("중복:", len(ids) - len(set(ids)))

총 세트: 7363
4K 포함 세트: 5882
중복: 0


In [3]:
# 전체에 돌려 실패율 확인 (대화형/노트북)
from pathlib import Path
from src.data.chart_parser import parse_osu
ok, fail = 0, []
for p in Path("data/raw").glob("*/*.osu"):
    try:
        parse_osu(p); ok += 1
    except Exception as e:
        fail.append((p, repr(e)))
print(f"성공 {ok} / 실패 {len(fail)}")
for p, e in fail[:10]: print(p, e)

성공 18452 / 실패 0


In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
import json
from pathlib import Path

metadata_path = Path("data/metadata/chart_metadata.csv")

if metadata_path.is_file():
    df = pd.read_csv(metadata_path)
else:
    source_path = Path("data/metadata/beatmapsets.jsonl")
    if not source_path.is_file():
        raise FileNotFoundError(
            "Neither chart_metadata.csv nor beatmapsets.jsonl was found."
        )

    with source_path.open(encoding="utf-8") as f:
        sets = [json.loads(line) for line in f if line.strip()]

    df = pd.DataFrame(
        [
            {
                **beatmap,
                "beatmapset_id": beatmapset.get("id"),
                "artist": beatmapset.get("artist"),
                "title": beatmapset.get("title"),
                "creator": beatmapset.get("creator"),
            }
            for beatmapset in sets
            for beatmap in beatmapset.get("beatmaps", [])
        ]
    )

    # osu! API calls this field "difficulty_rating"; normalize it to "sr".
    if "sr" not in df.columns and "difficulty_rating" in df.columns:
        df = df.rename(columns={"difficulty_rating": "sr"})

if "sr" not in df.columns:
    raise KeyError(f"'sr' column not found. Available columns: {df.columns.tolist()}")

df["sr"] = pd.to_numeric(df["sr"], errors="coerce")

sr_bins = [0, 2, 2.7, 4, 5.3, 6.5, np.inf]
df["sr_interval"] = pd.cut(df["sr"], bins=sr_bins, right=False)

# DataFrame rows and their original metadata indices, grouped by SR range.
charts_by_sr = {
    interval: group.copy()
    for interval, group in df.groupby("sr_interval", observed=True)
}
chart_indices_by_sr = {
    interval: group.index.tolist()
    for interval, group in charts_by_sr.items()
}

# Fetch all chart metadata with SR in [2.7, 4.0).
interval = pd.Interval(2.7, 4.0, closed="left")
display(charts_by_sr.get(interval, pd.DataFrame()))

# ..

grade_labels = ["Easy", "Normal", "Hard", "Insane", "Expert", "Expert+"]

df["sr_grade"] = pd.cut(
    df["sr"],
    bins=[0, 2, 2.7, 4, 5.3, 6.5, np.inf],
    labels=grade_labels,
    right=False,
    include_lowest=True,
)

# nps, notes_per_beat, long_note_ratio가 계산된 뒤 실행.

summary_columns = {
    "nps": "NPS 중앙값",
    "notes_per_beat": "비트당 노트 중앙값",
    "long_note_ratio": "롱노트 비율 중앙값",
}

aggregations = {"chart_count": ("sr", "size")}
aggregations.update(
    {
        f"median_{column}": (column, "median")
        for column in summary_columns
        if column in df.columns
    }
)

sr_summary = (
    df.groupby("sr_grade", observed=False)
    .agg(**aggregations)
    .reindex(grade_labels)
    .rename(
        columns={
            "chart_count": "채보 수",
            **{
                f"median_{column}": label
                for column, label in summary_columns.items()
                if column in df.columns
            },
        }
    )
)

missing_metrics = [column for column in summary_columns if column not in df.columns]
if missing_metrics:
    print(
        "다음 지표는 아직 계산되지 않아 표에서 제외되었습니다:",
        ", ".join(missing_metrics),
    )

sr_summary

,beatmapset_id,sr,id,lazer_only,mode,status,total_length,user_id,version,accuracy,...,passcount,playcount,ranked,url,checksum,max_combo,artist,title,creator,sr_interval
15,2377592,3.20567,5136746,False,mania,ranked,137,10400730,[7K] Hard,7.4,...,86,154,1,https://osu.ppy.sh/beatmaps/5136746,b977a6f4e0612ee7fca75c39fb17e8b9,1423,Lime,Stellaria,-NoName-,"[2.7, 4.0)"
29,2555651,3.56097,5676535,False,mania,ranked,111,37767001,[4K] Fragments,8.0,...,323,874,1,https://osu.ppy.sh/beatmaps/5676535,a53333844d06575ff00f3ea921e2db7b,1715,Neko Hacker,Shadows feat. Such,-Zell,"[2.7, 4.0)"
30,2555651,2.70506,5687874,False,mania,ranked,111,14938264,[4K] mirac1e's Hard,7.5,...,225,465,1,https://osu.ppy.sh/beatmaps/5687874,242aedcb3dfc262fb0855f788760132f,1386,Neko Hacker,Shadows feat. Such,-Zell,"[2.7, 4.0)"
33,2588771,3.47362,5775434,False,mania,ranked,94,6185083,[4K] Menphiss' Insane,7.5,...,436,743,1,https://osu.ppy.sh/beatmaps/5775434,a23aa9de997c67869159abea7816eedf,997,Noisia,Shellshock (feat. Foreign Beggars) (Cut Ver.),-Enma-,"[2.7, 4.0)"
34,2588771,2.76197,5775435,False,mania,ranked,94,34587251,[4K] Hard,7.0,...,333,607,1,https://osu.ppy.sh/beatmaps/5775435,160a748f0c619caa7a6d35fefe44dd50,790,Noisia,Shellshock (feat. Foreign Beggars) (Cut Ver.),-Enma-,"[2.7, 4.0)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26696,65847,3.45395,222331,False,mania,ranked,148,73480,7K MX,7.0,...,26901,60688,1,https://osu.ppy.sh/beatmaps/222331,2440e49ea778295ece061f4779cdc652,2147,nayuta,Toki wo Kizamu Uta,Hanyuu,"[2.7, 4.0)"
26707,74779,3.57273,217675,False,mania,ranked,120,146538,[7K] Insane,7.0,...,65194,134576,1,https://osu.ppy.sh/beatmaps/217675,57cc7bc1e851fd32da262d332fa32431,1389,Y&Co. feat. Karin,Sweet Rain,chonicle,"[2.7, 4.0)"
26711,71255,3.26466,204361,False,mania,ranked,197,277044,[7K] Insane,7.0,...,49515,100528,1,https://osu.ppy.sh/beatmaps/204361,17684598a6832403cc0e1ad04cf136a8,2881,sakuzyo,ChaiN De/structioN (siilento's solid remix),Entozer,"[2.7, 4.0)"
26715,63089,3.13765,193128,False,mania,ranked,91,2363,4K MX,7.0,...,291559,851904,1,https://osu.ppy.sh/beatmaps/193128,383029a705f65cfca86b7e3cb9df2e51,1306,fripSide,only my railgun (TV Size),DJPop,"[2.7, 4.0)"


다음 지표는 아직 계산되지 않아 표에서 제외되었습니다: nps, notes_per_beat, long_note_ratio


,채보 수
sr_grade,
Easy,6874
Normal,4752
Hard,7892
Insane,5004
Expert,1430
Expert+,768


In [8]:
from pathlib import Path

# 4K mania 채보만 분석한다.
if "mode_int" in df.columns and "cs" in df.columns:
    df = df[
        (pd.to_numeric(df["mode_int"], errors="coerce") == 3)
        & (pd.to_numeric(df["cs"], errors="coerce") == 4)
    ].copy()

metadata_id_column = next(
    (column for column in ("id", "beatmap_id") if column in df.columns),
    None,
)
if metadata_id_column is None:
    raise KeyError(f"Beatmap ID column not found: {df.columns.tolist()}")

target_ids = {
    int(value)
    for value in pd.to_numeric(df[metadata_id_column], errors="coerce").dropna()
}

cache_path = Path("data/metadata/chart_metrics.csv")
metric_columns = ["beatmap_id", "nps", "notes_per_beat", "long_note_ratio"]

if cache_path.is_file():
    metrics = pd.read_csv(cache_path)
    metrics = metrics[[column for column in metric_columns if column in metrics.columns]]
else:
    metrics = pd.DataFrame(columns=metric_columns)

cached_ids = set(
    pd.to_numeric(metrics.get("beatmap_id"), errors="coerce").dropna().astype(int)
)
missing_ids = target_ids - cached_ids


def beat_count_between(start_ms, end_ms, timing_points):
    """빨간 타이밍 포인트(BPM 변화)를 반영해 두 시점 사이의 박 수를 계산한다."""
    if end_ms <= start_ms or not timing_points:
        return np.nan

    active_beat_length = timing_points[0][1]
    for offset, beat_length in timing_points:
        if offset <= start_ms:
            active_beat_length = beat_length
        else:
            break

    beats = 0.0
    cursor = start_ms

    for offset, beat_length in timing_points:
        if offset <= start_ms:
            continue
        if offset >= end_ms:
            break

        beats += (offset - cursor) / active_beat_length
        cursor = offset
        active_beat_length = beat_length

    beats += (end_ms - cursor) / active_beat_length
    return beats


def parse_mania_metrics(path):
    """4K mania .osu 파일 하나에서 채보 지표를 계산한다."""
    section = None
    beatmap_id = None
    mode = None
    circle_size = None
    timing_points = []
    note_times = []
    hold_count = 0

    with path.open(encoding="utf-8-sig", errors="replace") as file:
        for raw_line in file:
            line = raw_line.strip()
            if not line:
                continue

            if line.startswith("[") and line.endswith("]"):
                section = line[1:-1]
                continue

            if section in {"Metadata", "General", "Difficulty"} and ":" in line:
                key, value = line.split(":", 1)
                key, value = key.strip(), value.strip()

                if section == "Metadata" and key == "BeatmapID":
                    beatmap_id = int(value)
                elif section == "General" and key == "Mode":
                    mode = int(value)
                elif section == "Difficulty" and key == "CircleSize":
                    circle_size = float(value)

            elif section == "TimingPoints":
                parts = line.split(",")
                if len(parts) >= 2:
                    # uninherited=1인 빨간 타이밍 포인트만 BPM 계산에 사용
                    is_red_point = len(parts) < 7 or parts[6] == "1"
                    if is_red_point:
                        try:
                            timing_points.append((float(parts[0]), float(parts[1])))
                        except ValueError:
                            pass

            elif section == "HitObjects":
                parts = line.split(",")
                if len(parts) < 4:
                    continue

                try:
                    time_ms = float(parts[2])
                    object_type = int(parts[3])
                except ValueError:
                    continue

                note_times.append(time_ms)

                # mania hold note는 type의 128 비트가 설정되어 있다.
                if object_type & 128:
                    hold_count += 1

    if beatmap_id is None or mode != 3 or circle_size != 4:
        return None

    note_times.sort()
    timing_points.sort()

    note_count = len(note_times)
    if note_count < 2:
        return {
            "beatmap_id": beatmap_id,
            "nps": np.nan,
            "notes_per_beat": np.nan,
            "long_note_ratio": hold_count / note_count if note_count else np.nan,
        }

    start_ms, end_ms = note_times[0], note_times[-1]
    duration_seconds = (end_ms - start_ms) / 1000
    beat_count = beat_count_between(start_ms, end_ms, timing_points)

    return {
        "beatmap_id": beatmap_id,
        # 첫 노트와 마지막 노트의 시작 시각 사이 기준
        "nps": note_count / duration_seconds if duration_seconds > 0 else np.nan,
        "notes_per_beat": note_count / beat_count if beat_count > 0 else np.nan,
        "long_note_ratio": hold_count / note_count,
    }


if missing_ids:
    rows = []
    osu_paths = list(Path("data/raw").rglob("*.osu"))

    for path in osu_paths:
        try:
            result = parse_mania_metrics(path)
            if result is not None and result["beatmap_id"] in missing_ids:
                rows.append(result)
        except Exception as error:
            print(f"Skipped {path}: {error}")

    new_metrics = pd.DataFrame(rows, columns=metric_columns)
    metrics = pd.concat([metrics, new_metrics], ignore_index=True)
    metrics = metrics.drop_duplicates("beatmap_id", keep="last")
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(cache_path, index=False)

metrics["beatmap_id"] = pd.to_numeric(metrics["beatmap_id"], errors="coerce")
metrics = metrics.dropna(subset=["beatmap_id"]).set_index("beatmap_id")

df["_beatmap_id"] = pd.to_numeric(df[metadata_id_column], errors="coerce")
for column in ("nps", "notes_per_beat", "long_note_ratio"):
    df[column] = df["_beatmap_id"].map(metrics[column])

# SR 등급 및 최종 표를 다시 생성한다.
grade_labels = ["Easy", "Normal", "Hard", "Insane", "Expert", "Expert+"]
df["sr_grade"] = pd.cut(
    df["sr"],
    bins=[0, 2, 2.7, 4, 5.3, 6.5, np.inf],
    labels=grade_labels,
    right=False,
    include_lowest=True,
)

sr_summary = (
    df.groupby("sr_grade", observed=False)
    .agg(
        **{
            "채보 수": ("sr", "size"),
            "NPS 중앙값": ("nps", "median"),
            "비트당 노트 중앙값": ("notes_per_beat", "median"),
            "롱노트 비율 중앙값": ("long_note_ratio", "median"),
        }
    )
    .reindex(grade_labels)
)

print(f"지표 계산 완료: {df['nps'].notna().sum()} / {len(df)}개 4K 채보")
sr_summary

지표 계산 완료: 18436 / 18453개 4K 채보


,채보 수,NPS 중앙값,비트당 노트 중앙값,롱노트 비율 중앙값
sr_grade,,,,
Easy,4620,3.743439,1.387674,0.152778
Normal,3472,6.435841,2.299174,0.148333
Hard,5903,9.546314,3.373786,0.150803
Insane,3581,12.901792,4.156002,0.130802
Expert,681,15.757067,4.522134,0.170055
Expert+,196,17.392655,4.657634,0.411215
